# 2주차 예제 — 따릉이 대여량 예측 (Linear Regression)

1주차에서는 데이터를 관찰하며 질문에 답하는 EDA를 익혔습니다. 이번 주에는 그 관찰을 실제 **예측 모델**에 반영합니다.

1. 선형회귀란 무엇인가 (간단한 예제로 원리부터)
2. 따릉이 데이터 준비
3. 변수 선택 — 어떤 변수로 예측합니까?
4. 단순선형회귀 — 추세 하나로만 예측
5. 다중선형회귀 — 계절/요일까지 반영
6. 잔차 분석 — 모델이 놓친 부분 확인
7. 인사이트 정리


## Part 1. 선형회귀란 무엇인가 (간단한 예제)

**선형회귀(Linear Regression)** 는 데이터 점들을 가장 잘 지나가는 **직선 하나**를 찾는 방법입니다. 변수 하나로 예측할 때는 이렇게 씁니다.

$$y = w \cdot x + b$$

- `w` (기울기, **계수/coefficient**): x가 1 늘어날 때 y가 얼마나 변하는지
- `b` (**절편/intercept**): x가 0일 때 y의 값

공부시간(x)으로 시험점수(y)를 예측하는 작은 예제로 원리를 직접 확인합니다.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.metrics import root_mean_squared_error

plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False


In [ ]:
toy = pd.DataFrame({
    '공부시간': [1, 2, 3, 4, 5, 6],
    '시험점수': [42, 50, 55, 68, 72, 85],
})

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(toy['공부시간'], toy['시험점수'])
ax.set_xlabel('공부시간')
ax.set_ylabel('시험점수')
ax.set_title('공부시간과 시험점수')
plt.show()


점들이 대체로 오른쪽 위로 올라가는 모양입니다. 직선 하나로 이 흐름을 표현해 봅니다. `LinearRegression().fit(X, y)`는 학습 데이터의 잔차 제곱합이 가장 작은 직선을 찾습니다.


In [ ]:
X = toy[['공부시간']]
y = toy['시험점수']

model = LinearRegression().fit(X, y)
print('기울기(w):', model.coef_[0])
print('절편(b):', model.intercept_)


In [ ]:
pred = model.predict(X)

fig, ax = plt.subplots(figsize=(5, 4))
ax.scatter(toy['공부시간'], toy['시험점수'], label='실제 값')
ax.plot(toy['공부시간'], pred, color='red', label='회귀선')
ax.set_xlabel('공부시간')
ax.set_ylabel('시험점수')
ax.legend()
plt.show()


빨간 선은 `model.coef_`(기울기)와 `model.intercept_`(절편)로 그린 직선입니다. '오차가 가장 작다'는 표현의 의미를 잔차로 확인합니다.


### 잔차와 RMSE — '오차가 가장 작다'는 게 무슨 뜻인가

**잔차(residual)**는 실제값에서 예측값을 뺀 값으로, 각 점이 회귀선에서 얼마나 떨어져 있는지를 나타냅니다. 잔차를 그대로 더하면 양수와 음수가 서로 상쇄되므로 오차의 크기를 나타내기 어렵습니다. 그래서 잔차를 **제곱해 평균**한 값을 **MSE(평균제곱오차)**로 사용합니다. MSE에 다시 제곱근을 적용한 값이 **RMSE**이며, 원래 목표값 y와 단위가 같아 오차 크기를 해석하기 쉽습니다.


In [ ]:
residuals = y - pred
print('잔차:', residuals.values)
print('잔차의 합:', residuals.sum().round(6), '← 거의 0 (선형회귀의 특성상 양/음이 상쇄됨)')

mse = (residuals ** 2).mean()
rmse = np.sqrt(mse)
print('\nMSE (직접 계산):', mse)
print('RMSE (직접 계산):', rmse)
print('RMSE (sklearn):', root_mean_squared_error(y, pred))


**직접 계산한 값과 `root_mean_squared_error()` 결과가 같습니다.** 이를 통해 RMSE가 잔차 제곱의 평균에 제곱근을 적용한 값임을 확인할 수 있습니다. 이어서 이 직선으로 7시간 공부했을 때의 점수를 계산합니다.


In [ ]:
new_pred = model.predict(pd.DataFrame({'공부시간': [7]}))
print('7시간 공부 예측 점수:', new_pred[0])


### 변수가 여러 개라면 어떻게 됩니까? — 다중선형회귀

변수가 하나면 직선, 변수가 여러 개면 다음처럼 각 변수마다 계수가 붙습니다.

$$y = w_1 x_1 + w_2 x_2 + \cdots + b$$

원리는 같습니다. 여러 계수 가운데 학습 데이터의 잔차 제곱합이 가장 작아지는 조합을 찾습니다. Part 5에서 실제 데이터로 이 확장 과정을 확인합니다.


## Part 2. 따릉이 데이터 준비

`일별 대여건수` 파일은 2024년 1월부터 2026년 6월까지 다섯 개로 나뉘어 있습니다. 원본은 [서울 열린데이터광장 — 서울시 공공자전거 따릉이 이용현황(일별 대여건수)](https://data.seoul.go.kr/dataList/OA-14994/A/1/datasetView.do)에서 내려받을 수 있습니다. 이 노트북은 `dataset/extracted/따릉이 공공데이터/02_이용정보/`에 준비된 `서울특별시 공공자전거 일별 대여건수_*.csv` 파일을 하나로 합쳐 912일의 연속된 시계열을 만듭니다.


In [ ]:
import glob

base = '../dataset/extracted/따릉이 공공데이터/02_이용정보'
files = sorted(glob.glob(base + '/서울특별시 공공자전거 일별 대여건수_*.csv'))
files


In [ ]:
dfs = [pd.read_csv(f, encoding='cp949') for f in files]
df = pd.concat(dfs, ignore_index=True)
df['대여일자'] = pd.to_datetime(df['대여일자'])
df = df.sort_values('대여일자').reset_index(drop=True)
df.shape


In [ ]:
# 빠진 날짜나 중복 날짜가 있으면 이후 분석이 왜곡되므로 먼저 확인
print('중복 날짜 수:', df['대여일자'].duplicated().sum())
print('기간:', df['대여일자'].min(), '~', df['대여일자'].max())
print('전체 일수:', len(pd.date_range(df["대여일자"].min(), df["대여일자"].max())), '/ 실제 행 수:', len(df))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df['대여일자'], df['대여건수'])
ax.set_title('일별 대여건수 (2024.01 ~ 2026.06)')
plt.show()


그래프에서는 두 가지 특징을 관찰할 수 있습니다. **① 매년 비슷한 패턴으로 오르내리는 계절성**, **② 겨울마다 대여건수가 크게 낮아지는 구간**입니다. 다음 단계에서는 이 패턴을 모델이 사용할 수 있는 변수로 표현합니다.


## Part 3. 변수 선택 — 어떤 변수로 예측합니까?

회귀모델에 사용할 변수는 의미와 예측 시점의 가용성을 살펴본 뒤 고릅니다. 여기서는 **대여건수와 관계가 있어 보이는 변수**를 그래프로 먼저 확인합니다.


In [ ]:
df['day_index'] = (df['대여일자'] - df['대여일자'].min()).dt.days  # 추세(trend)
df['월'] = df['대여일자'].dt.month
df['요일'] = df['대여일자'].dt.day_name()
df['주말여부'] = df['대여일자'].dt.dayofweek.isin([5, 6]).astype(int)

split_idx = len(df) - 60
train, test = df.iloc[:split_idx], df.iloc[split_idx:]
df[['대여일자', 'day_index', '월', '요일', '주말여부']].head()


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
train.groupby('월')['대여건수'].mean().plot(kind='bar', ax=ax)
ax.set_title('학습 구간의 월별 평균 대여건수')
plt.show()


In [ ]:
order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
fig, ax = plt.subplots(figsize=(7, 4))
train.groupby('요일')['대여건수'].mean().reindex(order).plot(kind='bar', ax=ax)
ax.set_title('학습 구간의 요일별 평균 대여건수')
plt.show()


마지막 60일을 먼저 비교 구간으로 남겨 두고, 앞선 852일의 학습 구간에서만 월·요일 평균을 확인했습니다. 두 그래프에서 차이가 보이므로 월과 요일을 회귀 입력 후보에 포함합니다. `day_index`는 전체적인 증가·감소 추세를 표현하기 위한 변수로 함께 사용합니다.


## Part 4. 단순선형회귀 — 추세 하나로만 예측

Part 1의 간단한 예제와 완전히 같은 방법입니다. 이번에는 `day_index`(며칠째인지) 하나만으로 대여건수를 예측합니다.

**train과 test는 시간 순서로 나눕니다.** 무작위로 섞으면 미래 데이터가 학습에 포함되고 계절적으로 가까운 날짜가 양쪽에 놓여, 실제 미래 예측보다 성능이 좋아 보일 수 있습니다. 이를 줄이기 위해 과거 구간으로 학습하고 마지막 60일을 미래 평가 구간으로 사용합니다.


In [ ]:
split_idx = len(df) - 60
train, test = df.iloc[:split_idx], df.iloc[split_idx:]
y_train, y_test = train['대여건수'], test['대여건수']
print('train:', train.shape, '/ test:', test.shape)


In [ ]:
simple_model = LinearRegression()
simple_model.fit(train[['day_index']], y_train)
pred_simple = simple_model.predict(test[['day_index']])

rmse_simple = root_mean_squared_error(y_test, pred_simple)
print('단순회귀 RMSE:', rmse_simple)
print('기울기(day_index 계수):', simple_model.coef_[0])


**계수 해석**: 기울기가 음수로 나옵니다. 이는 이 단순한 직선이 학습 기간에서 포착한 평균 기울기이며, 시간이 하루 흐르는 것이 대여 감소의 원인이라는 뜻은 아닙니다. 이 모델은 계절과 요일 정보를 사용하지 않아 매년 반복되는 오르내림을 충분히 반영하기 어렵습니다. 실제 값과 예측값을 함께 그려 차이를 확인합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test['대여일자'], y_test.values, label='실제')
ax.plot(test['대여일자'], pred_simple, label='단순회귀 예측')
ax.legend()
ax.set_title('단순선형회귀: 실제 vs 예측')
plt.xticks(rotation=45)
plt.show()


## Part 5. 다중선형회귀 — 계절/요일까지 반영

Part 1 끝에서 본 $y = w_1 x_1 + w_2 x_2 + \cdots + b$ 형태입니다. Part 3에서 확인한 `월`과 `요일`을 추가합니다. 두 변수는 범주형이므로 `pd.get_dummies`를 사용해 0/1 더미변수로 바꿉니다. 모든 범주 더미와 절편을 함께 넣으면 변수들이 서로를 완벽하게 설명하는 **다중공선성**이 생겨 계수 해석이 불안정해질 수 있습니다. `drop_first=True`로 남긴 1월과 금요일이 기준값이 되고, 나머지 계수는 그 기준과의 차이로 해석합니다.


In [ ]:
month_dummies = pd.get_dummies(df['월'], prefix='월', drop_first=True)
weekday_dummies = pd.get_dummies(df['요일'], prefix='요일', drop_first=True)
X = pd.concat([df[['day_index']], month_dummies, weekday_dummies], axis=1)
X.head()


In [ ]:
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]

multi_model = LinearRegression()
multi_model.fit(X_train, y_train)
pred_multi = multi_model.predict(X_test)

rmse_multi = root_mean_squared_error(y_test, pred_multi)
print('다중회귀 RMSE:', rmse_multi)
print('단순회귀 대비 개선율:', round((1 - rmse_multi / rmse_simple) * 100, 1), '%')


**계수 해석**: 계수가 큰 변수와 작은 변수를 몇 개 살펴보고, 기준 범주와 비교해 모델의 예측값이 어느 방향으로 달라지는지 확인합니다.


In [ ]:
coef_series = pd.Series(multi_model.coef_, index=X_train.columns).sort_values()
print('대여건수를 가장 많이 줄이는 요인 (기준 대비):')
print(coef_series.head())
print()
print('대여건수를 가장 많이 늘리는 요인 (기준 대비):')
print(coef_series.tail())


일요일·토요일 계수가 크게 음수로 나온다면, 다른 입력을 고정했을 때 기준인 금요일보다 주말에 모델의 예측값이 낮아진다는 뜻입니다. 이는 출퇴근 이용 가설과 잘 맞지만, 이용 목적을 직접 측정한 결과는 아닙니다. 봄·가을에 해당하는 월의 계수가 크게 양수인 결과도 온화한 날씨 가설과 일치하지만, 월이 여러 계절 요인을 함께 나타낸다는 한계를 고려합니다.


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test['대여일자'], y_test.values, label='실제')
ax.plot(test['대여일자'], pred_simple, label='단순회귀 예측', linestyle='--')
ax.plot(test['대여일자'], pred_multi, label='다중회귀 예측')
ax.legend()
ax.set_title('단순 vs 다중회귀: 실제 값과 비교')
plt.xticks(rotation=45)
plt.show()


## Part 6. 잔차 분석

Part 1에서 배운 잔차 개념을 실제 데이터에 적용합니다. 잔차가 **0 주변에서 뚜렷한 구조 없이 흩어지는 모습**은 모델을 점검하는 중요한 기준입니다. 잔차에 패턴이 남아 있다면 모델이 아직 설명하지 못한 정보가 있을 가능성을 살펴봅니다.


In [ ]:
residuals_simple = y_test.values - pred_simple
residuals_multi = y_test.values - pred_multi

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)
axes[0].scatter(pred_simple, residuals_simple)
axes[0].axhline(0, color='red')
axes[0].set_title('단순회귀 잔차')
axes[0].set_xlabel('예측값')
axes[0].set_ylabel('잔차(실제-예측)')

axes[1].scatter(pred_multi, residuals_multi)
axes[1].axhline(0, color='red')
axes[1].set_title('다중회귀 잔차')
axes[1].set_xlabel('예측값')
plt.show()


단순회귀 잔차는 특정 구간에서 몰리거나 치우친 패턴이 보일 수 있습니다 — 계절을 반영하지 못했기 때문입니다. 다중회귀 잔차가 상대적으로 0 주변에 고르게 흩어져 있다면, 그만큼 모델이 데이터의 패턴을 더 잘 설명한다는 뜻입니다.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(residuals_multi, bins=15)
ax.set_title('다중회귀 잔차 분포')
plt.show()


## Part 7. 인사이트 정리 (예시)

- 추세만 사용한 단순회귀는 RMSE가 크고 실제 값의 반복적인 오르내림을 충분히 반영하지 못했습니다.
- 월·요일 더미를 추가한 다중회귀에서는 RMSE가 낮아졌습니다. 이 평가 구간에서 계절성과 요일 패턴이 예측에 도움을 주었음을 보여 줍니다.
- 주말 요일 계수의 음수와 봄·가을 월 계수의 양수는 통근과 날씨 가설에 잘 맞습니다. 다만 관측 데이터의 조건부 연관성이므로 인과관계로 해석하려면 추가 근거가 필요합니다.
- 다중회귀의 잔차에도 패턴이 남아 있다면 공휴일, 강수, 기온 같은 변수를 예측 시점의 가용성에 맞게 추가해 볼 수 있습니다.

감귤 착과량 예측 과제에서도 **변수 선택 → 단순·다중회귀 → RMSE 평가 → 계수 해석 → 잔차 확인** 순서로 진행하면 각 판단을 자연스럽게 연결할 수 있습니다.
